# 4장. pandas로 데이터에 질문하기

이 노트북은 데이터를 선택·필터링·정렬하고, 안전하게 병합한 뒤 완료 주문 기준 요약표를 만드는 실습 자료입니다.


## 제출 정보

- GitHub ID: `chocoemong1`
- 작성일: 2026-09-29
- 제출 URL: <https://github.com/chocoemong1/llm-data-analysis-study/blob/main/chapter04/chapter04.ipynb>

## 학습 목표

- 실제 컬럼명과 값의 종류를 확인한 뒤 필터링합니다.
- 수량과 단가로 주문 상세 금액을 계산합니다.
- `validate`와 `indicator`를 사용해 병합 결과를 검증합니다.
- 취소·환불 주문을 제외하고 완료 주문 기준 매출을 계산합니다.
- 개인정보를 최소화한 분석 결과를 저장합니다.


## 제출 기록 1. 질문과 분석 범위

**질문:** completed 주문 기준 카테고리별 금액은 어떻게 다르며, 가장 큰 카테고리는 무엇인가?

| 항목 | 기준 |
| --- | --- |
| 파일 | orders, order_items, products |
| 주요 컬럼 | order_status, order_id, product_id, quantity, unit_price, category |
| 금액 | `line_total = quantity × unit_price` |
| 포함 상태 | `completed` |
| 집계 단위 | category |

![질문과 분석 범위](images/step01_question.png)

- **관찰:** 주문 상태·상세 금액·카테고리가 서로 다른 파일에 있어 두 번의 병합이 필요하다.
- **판단:** 취소·환불을 제외한 completed 범위를 유지하고 병합 행 수와 총합을 검증한다.
- **업무 의미:** 계산 정의가 명확해야 보고서 간 KPI 차이를 줄일 수 있다.
- **한계:** 이 금액은 할인·세금·부분 환불·원가를 반영한 회계상 순매출이 아니다.

## 1. 프로젝트 루트와 실행 환경 확인

VS Code에서 Notebook을 실행하면 현재 작업 폴더가 프로젝트 루트 또는 `notebooks` 폴더일 수 있습니다. 아래 코드는 상위 폴더를 확인해 프로젝트 루트를 찾습니다.


In [1]:
from pathlib import Path
import sys

import pandas as pd


def find_course_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        for root in [candidate, candidate / "llm-data-analysis-course"]:
            if (
                (root / "requirements.txt").exists()
                and (root / "scripts").exists()
                and (root / "data" / "raw").exists()
            ):
                return root
    raise FileNotFoundError("공식 수업 저장소를 찾을 수 없습니다.")


def find_study_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if candidate.name == "llm-data-analysis-study":
            return candidate
    raise FileNotFoundError("개인 학습 저장소를 찾을 수 없습니다.")


PROJECT_ROOT = find_course_root(Path.cwd())
STUDY_ROOT = find_study_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data" / "raw"
REPORT_DIR = STUDY_ROOT / "chapter04" / "outputs"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Python 실행 파일:", sys.executable)
print("현재 작업 폴더:", Path.cwd())
print("공식 수업 저장소:", PROJECT_ROOT)
print("데이터 폴더:", DATA_DIR)
print("개인 결과 폴더:", REPORT_DIR)

Python 실행 파일: /home/gem/research/프로그램 모음/venvs/llm-data-analysis/bin/python
현재 작업 폴더: /home/gem/research/학교 수업/LLM 수업/llm-data-analysis-study/chapter04
공식 수업 저장소: /home/gem/research/학교 수업/LLM 수업/llm-data-analysis-course
데이터 폴더: /home/gem/research/학교 수업/LLM 수업/llm-data-analysis-course/data/raw
개인 결과 폴더: /home/gem/research/학교 수업/LLM 수업/llm-data-analysis-study/chapter04/outputs


## 2. 데이터 파일 확인과 불러오기

파일이 없다면 프로젝트 루트에서 `python scripts/generate_sample_data.py`를 먼저 실행하세요.


In [2]:
required_files = ['customers.csv', 'products.csv', 'orders.csv', 'order_items.csv']
missing_files = [name for name in required_files if not (DATA_DIR / name).exists()]

if missing_files:
    raise FileNotFoundError(
        '필요한 데이터 파일이 없습니다: ' + ', '.join(missing_files)
        + '. 프로젝트 루트에서 python scripts/generate_sample_data.py를 실행하세요.'
    )

customers = pd.read_csv(DATA_DIR / 'customers.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')
orders = pd.read_csv(DATA_DIR / 'orders.csv')
order_items = pd.read_csv(DATA_DIR / 'order_items.csv')

print('데이터 불러오기 완료')


데이터 불러오기 완료


## 3. 데이터 구조와 필수 컬럼 확인

LLM이 작성한 코드가 실제 컬럼명과 일치하는지 먼저 확인합니다.


In [3]:
datasets = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

expected_columns = {
    'customers': ['customer_id', 'gender', 'age', 'city'],
    'products': ['product_id', 'product_name', 'category', 'price'],
    'orders': ['order_id', 'customer_id', 'order_date', 'order_status'],
    'order_items': ['order_id', 'product_id', 'quantity', 'unit_price'],
}

for name, df in datasets.items():
    missing = [col for col in expected_columns[name] if col not in df.columns]
    print(name, df.shape, df.columns.tolist())
    if missing:
        raise KeyError(f'{name}에 필요한 컬럼이 없습니다: {missing}')


customers (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
products (100, 4) ['product_id', 'product_name', 'category', 'price']
orders (300, 5) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
order_items (764, 5) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']


## 4. 컬럼 선택과 행 필터링

이 저장소의 샘플 데이터는 도시명을 `서울`, `부산`처럼 한글로 생성합니다. 필터링 전에 실제 값을 확인합니다.


In [4]:
customer_basic = customers[['customer_id', 'gender', 'age', 'city']]
display(customer_basic.head())

print('도시별 고객 수')
display(customers['city'].value_counts())

customers_over_30 = customers[customers['age'] >= 30]
seoul_customers = customers[customers['city'] == '서울']
city_customers = customers[customers['city'].isin(['서울', '부산'])]

print('30세 이상 고객 수:', len(customers_over_30))
print('서울 고객 수:', len(seoul_customers))
print('서울 또는 부산 고객 수:', len(city_customers))


,customer_id,gender,age,city
0,1,F,19,광주
1,2,F,32,대구
2,3,F,61,성남
3,4,F,55,울산
4,5,F,19,부산


도시별 고객 수


city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64

30세 이상 고객 수: 111
서울 고객 수: 15
서울 또는 부산 고객 수: 31


## 5. 정렬과 주문 상태 확인


In [5]:
display(products.sort_values('price', ascending=False).head(10))
display(customers.sort_values('age', ascending=False).head(10))

print('주문 상태별 건수')
display(orders['order_status'].value_counts())


,product_id,product_name,category,price
98,99,뷰티 상품 099,뷰티,200000
69,70,패션 상품 070,패션,198000
57,58,식품 상품 058,식품,197000
42,43,뷰티 상품 043,뷰티,197000
23,24,스포츠 상품 024,스포츠,196000
8,9,스포츠 상품 009,스포츠,193000
36,37,뷰티 상품 037,뷰티,193000
71,72,뷰티 상품 072,뷰티,189000
7,8,스포츠 상품 008,스포츠,189000
52,53,생활용품 상품 053,생활용품,188000


,customer_id,name,gender,age,city,signup_date
14,15,장정식,M,69,서울,2026-06-04
8,9,송지민,M,69,서울,2025-10-19
54,55,윤영철,M,69,광주,2023-07-31
78,79,이서준,M,69,부산,2023-07-14
136,137,김선영,M,68,울산,2024-11-28
45,46,김서현,M,67,대전,2026-05-13
132,133,김성진,M,67,성남,2026-05-29
123,124,김시우,M,67,대구,2024-09-24
11,12,김정남,F,66,대전,2024-09-30
57,58,김성훈,F,66,성남,2025-06-14


주문 상태별 건수


order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

## 6. 주문 상세 금액 만들기

`line_total`은 주문 상세 1행의 금액입니다. 주문 상태를 연결하기 전 합계는 확정 매출이 아니라 전체 주문 상세 금액입니다.


In [6]:
order_items = order_items.copy()
order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

all_order_amount = order_items['line_total'].sum()
print('전체 주문 상세 금액:', all_order_amount)
display(order_items[['order_id', 'product_id', 'quantity', 'unit_price', 'line_total']].head())


전체 주문 상세 금액: 255610000


,order_id,product_id,quantity,unit_price,line_total
0,1,100,3,102000,306000
1,1,87,5,25000,125000
2,1,7,3,142000,426000
3,1,9,3,193000,579000
4,2,72,4,189000,756000


## 제출 기록 2. 필터링·정렬·파생 컬럼

- 30세 이상 고객 111명, 서울 고객 15명, 서울·부산 고객 31명
- 상품은 `price` 내림차순으로 정렬
- `line_total = quantity × unit_price`
- 주문 상태 연결 전 전체 주문상세 금액: 255,610,000

![필터·정렬·파생 컬럼](images/step02_transform.png)

- **관찰:** 파생 금액은 계산됐지만 이 단계에는 completed·cancelled·refunded가 섞여 있다.
- **판단:** 상태 연결 전 합계는 완료 주문 금액으로 사용하지 않는다.
- **업무 의미:** 같은 계산식도 포함 상태에 따라 KPI가 달라진다.
- **한계:** 할인·배송비·세금 정보가 없어 실제 정산 금액은 알 수 없다.

## 7. 주문 데이터 병합과 검증

`validate='many_to_one'`은 주문 상세의 동일 주문 ID는 여러 번 나올 수 있지만 주문 테이블의 주문 ID는 한 번만 나와야 한다는 뜻입니다.


In [7]:
print('orders.order_id 중복 수:', orders['order_id'].duplicated().sum())

order_sales = order_items.merge(
    orders[['order_id', 'customer_id', 'order_date', 'order_status']],
    on='order_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(order_items))
print('병합 후 행 수:', len(order_sales))
display(order_sales['_merge'].value_counts())

order_sales = order_sales.drop(columns='_merge')


orders.order_id 중복 수: 0
병합 전 행 수: 764
병합 후 행 수: 764


_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64

## 8. 완료 주문만 선택하기

이후의 매출 요약은 `order_status == 'completed'`인 주문만 사용합니다.


In [8]:
order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
print('날짜 변환 실패:', order_sales['order_date'].isna().sum())

completed_order_sales = order_sales[
    order_sales['order_status'] == 'completed'
].copy()

completed_order_sales['order_month'] = (
    completed_order_sales['order_date'].dt.to_period('M').astype(str)
)

print('전체 주문 상세 행 수:', len(order_sales))
print('완료 주문 상세 행 수:', len(completed_order_sales))
print('완료 주문 매출:', completed_order_sales['line_total'].sum())


날짜 변환 실패: 0
전체 주문 상세 행 수: 764
완료 주문 상세 행 수: 474
완료 주문 매출: 148990000


## 9. 상품 데이터 병합과 검증


In [9]:
print('products.product_id 중복 수:', products['product_id'].duplicated().sum())

completed_sales_items = completed_order_sales.merge(
    products,
    on='product_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(completed_order_sales))
print('병합 후 행 수:', len(completed_sales_items))
display(completed_sales_items['_merge'].value_counts())

completed_sales_items = completed_sales_items.drop(columns='_merge')


products.product_id 중복 수: 0
병합 전 행 수: 474
병합 후 행 수: 474


_merge
both          474
left_only       0
right_only      0
Name: count, dtype: int64

## 제출 기록 3. merge 검증

| 병합 | key 중복 | 전→후 행 수 | indicator |
| --- | ---: | ---: | --- |
| order_items → orders | 0 | 764→764 | both 764 |
| completed 주문상세 → products | 0 | 474→474 | both 474 |

두 병합 모두 `validate="many_to_one"`을 통과했다.

![merge 검증](images/step03_merge.png)

- **관찰:** 기준 키가 고유했고 행 수 증가·감소와 미매칭이 없었다.
- **판단:** 현재 키와 범위에서는 중복 집계 위험이 낮은 병합이다.
- **업무 의미:** 잘못된 다대다 병합은 금액을 실제보다 크게 만들 수 있다.
- **한계:** 참조 관계가 맞아도 각 값의 업무적 정확성까지 보장하지 않는다.

## 10. 카테고리별·상품별 매출


In [10]:
category_sales = (
    completed_sales_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
).round(2)

display(category_sales)

product_sales = (
    completed_sales_items
    .groupby(['product_id', 'product_name', 'category'], as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

display(product_sales.head(10))


,category,total_quantity,total_sales,sales_ratio
3,스포츠,295,31743000,21.31
5,전자기기,259,26400000,17.72
2,생활용품,272,23915000,16.05
1,뷰티,223,23383000,15.69
4,식품,133,16573000,11.12
0,도서,149,16389000,11.00
6,패션,111,10587000,7.11


,product_id,product_name,category,total_quantity,total_sales
39,41,스포츠 상품 041,스포츠,35,5705000
11,12,식품 상품 012,식품,25,4375000
8,9,스포츠 상품 009,스포츠,20,3860000
70,72,뷰티 상품 072,뷰티,20,3780000
69,71,전자기기 상품 071,전자기기,23,3703000
66,68,스포츠 상품 068,스포츠,26,3640000
78,81,전자기기 상품 081,전자기기,22,3630000
10,11,패션 상품 011,패션,31,3565000
20,22,생활용품 상품 022,생활용품,29,3248000
86,89,생활용품 상품 089,생활용품,30,3090000


## 11. 월별 매출


In [11]:
monthly_summary = (
    completed_order_sales
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
)

monthly_summary['average_order_value'] = (
    monthly_summary['total_sales'] / monthly_summary['order_count']
).round(0)

display(monthly_summary)


,order_month,total_sales,order_count,average_order_value
0,2025-07,5869000,8,733625.0
1,2025-08,15621000,18,867833.0
2,2025-09,10190000,13,783846.0
3,2025-10,25766000,26,991000.0
4,2025-11,8812000,12,734333.0
5,2025-12,11501000,14,821500.0
6,2026-01,17423000,22,791955.0
7,2026-02,9749000,17,573471.0
8,2026-03,13429000,14,959214.0
9,2026-04,17553000,23,763174.0


## 12. 고객별 구매 금액

고객 ID로 먼저 집계한 뒤 고객 속성을 붙입니다. 출력에는 실제 이름 대신 익명화된 고객 라벨을 사용합니다.


In [12]:
customer_sales = (
    completed_order_sales
    .groupby('customer_id', as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

customer_sales = customer_sales.merge(
    customers[['customer_id', 'city']],
    on='customer_id',
    how='left',
    validate='one_to_one',
)

customer_sales['customer_label'] = 'Customer ' + customer_sales['customer_id'].astype(str)
display(customer_sales[['customer_label', 'city', 'order_count', 'total_sales']].head(10))


,customer_label,city,order_count,total_sales
0,Customer 117,성남,5,4100000
1,Customer 102,고양,4,3996000
2,Customer 83,수원,4,3880000
3,Customer 30,서울,5,3590000
4,Customer 40,서울,4,3523000
5,Customer 20,인천,2,3191000
6,Customer 3,성남,2,3178000
7,Customer 111,광주,3,3153000
8,Customer 66,서울,4,3093000
9,Customer 147,부산,2,2990000


## 제출 기록 4. completed 주문 집계

- 완료 주문: 184건, 상세 474행, 고객 100명
- 완료 주문 기준 금액: 148,990,000
- 카테고리 1위: 스포츠 31,743,000(21.31%), 최하위: 패션 10,587,000
- 상품 1위: 스포츠 상품 041, 5,705,000
- 월 최고: 2025-10, 25,766,000 / 26건
- 고객 최고: Customer 117, 4,100,000

![핵심 집계 결과](images/step04_groupby.png)

- **관찰:** 스포츠의 금액과 수량이 가장 컸고 2025-10 월 금액이 가장 컸다.
- **판단:** 단가·수량·주문 수를 분리해 봐야 인기나 원인을 판단할 수 있다.
- **업무 의미:** 재고·프로모션 검토의 출발점으로 사용할 수 있다.
- **한계:** 첫·마지막 달은 불완전 월이며 금액만으로 수익성이나 인과를 단정할 수 없다.

In [13]:
base_total = completed_order_sales["line_total"].sum()
category_total = category_sales["total_sales"].sum()
product_total = product_sales["total_sales"].sum()
monthly_total = monthly_summary["total_sales"].sum()
customer_total = customer_sales["total_sales"].sum()

total_check = pd.DataFrame({
    "scope": ["base", "category", "product", "month", "customer"],
    "total_sales": [
        base_total,
        category_total,
        product_total,
        monthly_total,
        customer_total,
    ],
})
total_check["difference_from_base"] = total_check["total_sales"] - base_total
display(total_check)
assert (total_check["difference_from_base"] == 0).all()

,scope,total_sales,difference_from_base
0,base,148990000,0
1,category,148990000,0
2,product,148990000,0
3,month,148990000,0
4,customer,148990000,0


## 제출 기록 5. 총합 일치 검증

원본 completed, 카테고리, 상품, 월, 고객 합계가 모두 **148,990,000**이며 차이는 0이었다.

![총합 일치 검증](images/step05_total_check.png)

- **판단:** 동일 범위의 총합 일치는 현재 확인 가능한 누락·중복 가능성이 낮다는 근거다.
- **불일치 시 확인:** completed 필터 → line_total 계산 → merge 행 수·미매칭 → 결측 group key → 타입 순으로 점검한다.

## 13. 결과 저장하기


In [14]:
outputs = {
    'ch04_category_sales.csv': category_sales,
    'ch04_product_sales.csv': product_sales,
    'ch04_monthly_sales.csv': monthly_summary,
    'ch04_customer_sales.csv': customer_sales,
}

for filename, df in outputs.items():
    path = REPORT_DIR / filename
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print(filename, path.exists(), path.stat().st_size)


ch04_category_sales.csv True 246
ch04_product_sales.csv True 4310
ch04_monthly_sales.csv True 428
ch04_customer_sales.csv True 3286


In [15]:
saved_category_sales = pd.read_csv(REPORT_DIR / 'ch04_category_sales.csv')
print('저장 결과 shape:', saved_category_sales.shape)
print('저장 결과 columns:', saved_category_sales.columns.tolist())
display(saved_category_sales.head())
assert saved_category_sales.shape == category_sales.shape
assert saved_category_sales['total_sales'].sum() == base_total


저장 결과 shape: (7, 4)
저장 결과 columns: ['category', 'total_quantity', 'total_sales', 'sales_ratio']


,category,total_quantity,total_sales,sales_ratio
0,스포츠,295,31743000,21.31
1,전자기기,259,26400000,17.72
2,생활용품,272,23915000,16.05
3,뷰티,223,23383000,15.69
4,식품,133,16573000,11.12


## 14. LLM 코드 검증 연습

```text
LLM이 다음 코드를 제안했습니다.

category_sales = order_items.groupby('category')['line_total'].sum()

현재 데이터에서 이 코드가 바로 실행 가능한지 검토해 주세요.
category 컬럼의 위치, 주문 상태 필터링, merge validate와 indicator를 포함해
안전한 수정 코드와 검증 순서를 설명해 주세요.
```


In [16]:
llm_validation = pd.DataFrame([
    {
        "check": "category 컬럼 위치",
        "proposal": "order_items에서 바로 groupby",
        "actual": "products에 존재",
        "decision": "수정",
    },
    {
        "check": "주문 상태",
        "proposal": "필터 없음",
        "actual": "completed만 포함",
        "decision": "수정",
    },
    {
        "check": "병합 검증",
        "proposal": "없음",
        "actual": "validate + indicator + 행 수",
        "decision": "추가",
    },
    {
        "check": "총합 검증",
        "proposal": "없음",
        "actual": "다섯 합계 일치",
        "decision": "추가",
    },
])
display(llm_validation)

,check,proposal,actual,decision
0,category 컬럼 위치,order_items에서 바로 groupby,products에 존재,수정
1,주문 상태,필터 없음,completed만 포함,수정
2,병합 검증,없음,validate + indicator + 행 수,추가
3,총합 검증,없음,다섯 합계 일치,추가


## 제출 기록 6. LLM 코드 검증

- **Prompt 요약:** completed 카테고리별 금액 계산과 실제 컬럼·키·병합·총합 검증 요청
- **제안 요약:** `order_items.groupby("category")`로 바로 합계
- **문제:** category가 없고 주문 상태 필터와 병합 검증도 없다.
- **수정:** orders·products를 검증하며 병합하고 completed만 집계한 뒤 총합을 대조
- **최종 판단:** 수정 후 사용

![LLM 코드 검증](images/step06_llm_validation.png)

코드가 실행돼도 범위·키·단위가 틀리면 답은 틀릴 수 있으므로 실제 컬럼, 상태값, 병합 행 수, 미매칭과 총합을 사람이 검증해야 한다.

## 15. 실습 과제

1. 40세 이상 고객을 추출합니다.
2. 상품 가격이 낮은 순서대로 10개를 출력합니다.
3. 결제수단별 주문 수와 비율을 계산합니다.
4. 카테고리별 평균 상품 가격을 계산합니다.
5. 완료 주문과 전체 주문의 금액 차이를 계산합니다.
6. 병합 검증을 포함한 고객별 구매 금액 코드 요청 프롬프트를 작성합니다.


In [17]:
# 과제 1. 40세 이상 고객
customers_40_plus = customers[customers["age"] >= 40]
print("40세 이상 고객 수:", len(customers_40_plus))

# 과제 2. 가격이 낮은 상품 10개
lowest_price_products = products.sort_values("price").head(10)
display(lowest_price_products[["product_id", "product_name", "category", "price"]])

# 과제 3. 결제수단별 주문 수와 비율
payment_summary = (
    orders["payment_method"]
    .value_counts()
    .rename_axis("payment_method")
    .reset_index(name="order_count")
)
payment_summary["ratio_pct"] = (
    payment_summary["order_count"] / len(orders) * 100
).round(2)
display(payment_summary)

# 과제 4. 카테고리별 평균 상품 가격
category_average_price = (
    products.groupby("category", as_index=False)
    .agg(average_price=("price", "mean"))
    .sort_values("average_price", ascending=False)
)
display(category_average_price)

# 과제 5. 전체 주문상세 금액과 완료 주문 금액 차이
amount_comparison = pd.DataFrame({
    "metric": ["all_order_detail_amount", "completed_amount", "difference"],
    "amount": [
        all_order_amount,
        completed_order_sales["line_total"].sum(),
        all_order_amount - completed_order_sales["line_total"].sum(),
    ],
})
display(amount_comparison)

# 과제 6. 병합 검증을 포함한 고객별 구매 금액 요청 Prompt
customer_sales_prompt = """
customers, orders, order_items로 completed 주문 기준 고객별 구매 금액을 계산해 주세요.
line_total은 quantity × unit_price이며 주문 수는 order_id.nunique()를 사용합니다.
orders.order_id 고유성을 먼저 확인하고 many_to_one merge와 indicator를 사용하세요.
병합 전후 행 수, 미매칭, 날짜 변환 실패, 원본 합계와 고객별 합계도 검증하세요.
고객 이름 등 개인정보는 출력하지 말고 익명 customer_id만 사용하세요.
"""
print(customer_sales_prompt.strip())

40세 이상 고객 수: 78


,product_id,product_name,category,price
5,6,전자기기 상품 006,전자기기,5000
27,28,뷰티 상품 028,뷰티,5000
97,98,스포츠 상품 098,스포츠,10000
46,47,생활용품 상품 047,생활용품,11000
94,95,전자기기 상품 095,전자기기,20000
49,50,뷰티 상품 050,뷰티,23000
82,83,전자기기 상품 083,전자기기,24000
86,87,도서 상품 087,도서,25000
41,42,패션 상품 042,패션,28000
58,59,뷰티 상품 059,뷰티,32000


,payment_method,order_count,ratio_pct
0,kakao_pay,79,26.33
1,naver_pay,77,25.67
2,bank_transfer,74,24.67
3,card,70,23.33


,category,average_price
4,식품,137142.857143
1,뷰티,117687.500000
6,패션,115909.090909
3,스포츠,111578.947368
0,도서,106857.142857
5,전자기기,101588.235294
2,생활용품,96437.500000


,metric,amount
0,all_order_detail_amount,255610000
1,completed_amount,148990000
2,difference,106620000


customers, orders, order_items로 completed 주문 기준 고객별 구매 금액을 계산해 주세요.
line_total은 quantity × unit_price이며 주문 수는 order_id.nunique()를 사용합니다.
orders.order_id 고유성을 먼저 확인하고 many_to_one merge와 indicator를 사용하세요.
병합 전후 행 수, 미매칭, 날짜 변환 실패, 원본 합계와 고객별 합계도 검증하세요.
고객 이름 등 개인정보는 출력하지 말고 익명 customer_id만 사용하세요.


## Chapter 04 최종 인사이트

1. 스포츠 카테고리가 completed 금액 31,743,000으로 가장 컸지만, 이것만으로 인기나 수익성이 가장 높다고 단정할 수 없다.
2. 전체 주문상세 255,610,000 중 completed 금액은 148,990,000으로, 상태를 무시하면 106,620,000만큼 과대 집계된다.

**추가 질문:** 카테고리 차이는 수량과 단가 중 무엇의 영향이 큰가? 월별 변화는 주문 수와 주문당 금액 중 무엇이 주도하는가?

**한계:** 샘플 데이터와 `quantity × unit_price`만 사용했으므로 실제 순매출·이익·프로모션 효과와 인과관계를 판단할 수 없다.

## 최종 제출 체크

- [x] 전체 코드 셀 실행 및 오류 0
- [x] 필터·파생 컬럼·두 merge·네 집계 수행
- [x] 총합 일치 및 저장 CSV 재확인
- [x] LLM 제안을 실제 데이터와 비교
- [x] Evidence 6장과 해석·한계 작성
- [x] 개인정보·Secret 없음
- [x] GitHub에서 Notebook과 이미지 표시 확인
- [ ] 최종 Notebook URL을 LMS에 제출

## 정리

이번 장에서는 실제 값 확인, 컬럼 선택, 필터링, 정렬, 파생 컬럼, 병합 검증, 완료 주문 기준 집계, CSV 저장 과정을 수행했습니다. 다음 장에서는 결측치, 중복, 타입 오류, 날짜 형식, 이상값 후보를 다루는 데이터 전처리로 이어집니다.
